# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessica245818/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I want to help a content editor decide which pages deserve limited review time first. This lane fits the available page-level search, engagement, age, and freshness measurements, and it leads to an operational output: a ranked review queue with suggested actions and reason codes. The goal is decision support, not automatic editing. I will keep a transparent rule-based queue as the baseline and use ML only if it improves top-of-list usefulness under honest client- and time-aware validation.

In [1]:
from pathlib import Path
import pandas as pd

# Find the repository root in both Colab and local runs.
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find the starter dataset from this working directory.")
    repo_root = repo_root.parent

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
assert df["content_id"].is_unique, "Expected one row per content item."
print(f"Loaded {len(df):,} content items from the public-safe starter dataset.")


Loaded 30,000 content items from the public-safe starter dataset.


## 2. The question: decision, action, cost of a wrong call

**Search question.** Among content items with enough observed exposure to support a decision, which should a content editor review first for refresh, metadata improvement, expansion, protection, or monitoring?

- **Unit of analysis:** one pseudonymized content item at a decision date. The starter dataset has one row per content item; later warehouse work will aggregate past daily observations to this same decision grain.
- **Decision and user:** a content editor chooses the next limited batch of pages to inspect—provisionally the top 50 per review cycle.
- **Output and action:** a ranked queue containing a priority score, confidence label, and observable reason codes. The editor inspects the evidence and then chooses an action; the system does not edit or prune automatically.
- **Cost of a wrong call:** a false positive wastes scarce editorial hours and could encourage unnecessary changes to a healthy page. A false negative leaves a meaningful decline or opportunity unattended. Because both errors matter, the initial success metric will be **Precision@50** alongside coverage/recall and manual review of borderline cases.
- **Why data or ML may help:** age, visibility, position, CTR, freshness, and engagement can interact in ways a single threshold misses. However, ML has to earn its place by beating an explainable rule baseline on unseen clients and later time windows. If it does not, the rule or a dashboard is the better product.

In [2]:
review_capacity = 50
decision_grain = "one content item at a decision date"
primary_metric = f"Precision@{review_capacity}"

print("Decision grain:", decision_grain)
print("Provisional review capacity:", review_capacity, "items per cycle")
print("Primary ranking metric:", primary_metric)


Decision grain: one content item at a decision date
Provisional review capacity: 50 items per cycle
Primary ranking metric: Precision@50


## 3. Quick look at the data (2-3 real numbers)

The starter slice contains **30,000 content items across 32 pseudonymized clients**, so a client-aware ranking problem is large enough to investigate. **16,262 items (54.2%)** are marked as declining in the current snapshot; that high base rate shows why “flag every decline” is not a useful action policy—a reviewer still needs prioritization. A more specific observable segment contains **9,759 pages** with at least 500 impressions, a measured average position from 1–20, and CTR below 0.5%; together they received **90,968,008 impressions** in the trailing window. This does not prove those pages need metadata changes, but it shows a substantial exposed population where a careful review queue could focus limited effort.

In [3]:
declining = df["trend_direction"].eq("down")
visible_low_ctr = (
    df["impressions_90d"].ge(500)
    & df["avg_position"].gt(0)  # zero means no position data
    & df["avg_position"].le(20)
    & df["ctr"].lt(0.5)         # rate columns are percentage points: 0.5 means 0.5%
)

summary = pd.Series({
    "content_items": len(df),
    "pseudonymized_clients": df["client_id"].nunique(),
    "declining_items": int(declining.sum()),
    "declining_share_pct": 100 * declining.mean(),
    "visible_low_ctr_candidates": int(visible_low_ctr.sum()),
    "candidate_impressions_90d": int(df.loc[visible_low_ctr, "impressions_90d"].sum()),
})

print(summary.to_string(float_format=lambda value: f"{value:,.1f}"))


content_items                    30,000.0
pseudonymized_clients                32.0
declining_items                  16,262.0
declining_share_pct                  54.2
visible_low_ctr_candidates        9,759.0
candidate_impressions_90d    90,968,008.0


## 4. Careful words: what I can and can't claim

This project may report **observed associations**, measured ranking performance, and directional evidence that a queue helps identify pages worth human review. It may say that a page is a review candidate because it has sufficient exposure plus specific measured signals. It cannot claim that those signals are Google ranking factors, that a recommended edit will cause recovery, or that a low CTR proves poor metadata or intent match. The starter `trend_direction` field is a current-window proxy, not an ideal future outcome, and neither it nor `trend_pct` may be used as model features. For the capstone I will prefer prior-window features and a later observed outcome, use client-grouped and time-aware validation, compare against a transparent baseline, and keep final recommendations subject to editorial review.

In [4]:
# Guardrails for later modeling: identifiers are grouping keys, and these two fields encode the proxy label.
excluded_from_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
assert excluded_from_features.issubset(df.columns)
print("Excluded from model features:", ", ".join(sorted(excluded_from_features)))
print("Claim type: observational, directional, and decision-support only.")


Excluded from model features: client_id, content_id, trend_direction, trend_pct
Claim type: observational, directional, and decision-support only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Saved under `work/notebooks/`; commit verification is completed with submission.